In [ ]:
# IPython magig  tools
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import time
# Plotting libraries
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from scipy.stats import pearsonr, ttest_rel, ttest_ind  
import seaborn as sns
import pandas as pd
import numpy as np
import datetime
from aind_vr_foraging_analysis.utils.plotting import plotting_friction_experiment as f
from aind_vr_foraging_analysis.utils.parsing import data_access, parse, AddExtraColumns
import aind_vr_foraging_analysis.utils.plotting as plotting

from matplotlib.backends.backend_pdf import PdfPages
sns.set_context('talk')

import warnings
pd.options.mode.chained_assignment = None  # Ignore SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

color1='#d95f02'
color2='#1b9e77'
color3='#7570b3'
# color4='yellow'
color4='#e7298a'

odor_list_color = [color1, color2, color3, color4]

pdf_path = r'Z:\scratch\vr-foraging\sessions'
base_path = r'Z:/scratch/vr-foraging/data/'
results_path = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\manuscript\results\figures'
data_path = r'C:\git\Aind.Behavior.VrForaging.Analysis\data'

palette = {
    'control': 'darkgrey',  # Red
    'friction_high': "#1e4110",  # Purple
    'friction_med': "#120c8a",  # Lighter Purple
    'friction_low': '#9e9ac8',  # Lightest Purple
    'distance_extra_short': 'crimson',  # Blue
    'distance_short': 'pink',  # Lighter Blue
    'distance_extra_long': '#fd8d3c',  # Yellow
    'distance_long': '#fdae6b'  # Lighter Yellow
}

color_dict_label = {'Ethyl Butyrate': color1, 'Alpha-pinene': color2, 'Amyl Acetate': color3, 
                    '2-Heptanone' : color2, 'Methyl Acetate': color1, 'Fenchone': color3, '2,3-Butanedione': color4,
                    'Methyl Butyrate': color1, '60': color2, '90': color1, "0": color3}

from matplotlib.backends.backend_pdf import PdfPages as pdf


In [ ]:
def plot_reward_probability(
    general_df,
    experiment: str = None,
    condition: str = 'session_n',
    odor_labels: list = ['Methyl Butyrate', 'Alpha-pinene'],
    save=False,
):
    # Filter
    general_df = general_df.loc[general_df.patch_label != '0'].copy()
    if experiment:
        col = 'experiment' if 'experiment' in general_df.columns else 'stage'
        general_df = general_df.loc[general_df[col] == experiment]

    fig, ax = plt.subplots(figsize=(3, 4))
    variable = 'reward_probability'

    # --- Boxplot ---
    sns.boxplot(x='patch_label', y=variable, palette=color_dict_label,
                data=general_df, order=odor_labels, zorder=10,
                width=0.5, ax=ax, fliersize=0,
                linewidth=0, 
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color='black', linewidth=1),
                capprops=dict(color='k', linewidth=1),
                flierprops=dict(marker=''))  # Example adjustment)

    # --- Connecting lines per session ---
    x_map = {label: i for i, label in enumerate(odor_labels)}
    for session, sdf in general_df.groupby(condition):
        pts = sdf.set_index('patch_label')[variable].reindex(odor_labels).dropna()
        if len(pts) == 2:
            ax.plot([x_map[l] for l in pts.index], pts.values,
                    color='black', alpha=0.4, linewidth=1, marker='')

    # --- Significance ---
    g1 = general_df.loc[general_df.patch_label == odor_labels[0], variable].dropna()
    g2 = general_df.loc[general_df.patch_label == odor_labels[1], variable].dropna()
    try:
        _, p = ttest_rel(g1, g2, nan_policy='omit')
    except:
        _, p = ttest_ind(g1, g2, nan_policy='omit')

    print(f"p-value for {odor_labels[0]} vs {odor_labels[1]}: {p:.4f}")
    if p < 0.001:   sig = '***'
    elif p < 0.01:  sig = '**'
    elif p < 0.05:  sig = '*'
    else:           sig = 'ns'

    y_top = general_df[variable].max()
    y_bar, h = y_top + 0.05, 0.025
    ax.plot([0, 0, 1, 1], [y_bar, y_bar + h, y_bar + h, y_bar], lw=1.2, color='black')
    ax.text(0.5, y_bar + h, sig, ha='center', va='bottom', fontsize=10)

    # --- Formatting ---
    ax.set_ylabel('P(reward) \n at leaving')
    ax.set_xlabel('')
    ax.set_ylim(0.3, 0.9)
    # ax.set_yticks([0, 0.5, 1])
    sns.despine(bottom=True)
    sns.despine()
    plt.tight_layout()
    if save:
        plt.savefig(save, format='pdf')
    else:
        plt.show()
        plt.close()

In [ ]:
# Recover and clean batch 4 dataset
data_path = r'../../data/'

# batch4 = pd.read_csv(data_path + 'batch_4.csv') # if you want the original dataset
batch4 = pd.read_csv(os.path.join(data_path, 'batch_4_fixed_interpatch.csv'))

# These mice are in the dataset but didn't perform the manipulation
batch4 = batch4[(batch4['mouse'] != 754573)&(batch4['mouse'] != 754572)&(batch4['mouse'] != 745300)&(batch4['mouse'] != 745306)&(batch4['mouse'] != 745307)]

batch4["session"] = batch4["session"].apply(lambda x: str(x).split('_')[-1])

## Micr with weird behavior
# batch4 = batch4.loc[(batch4.mouse != 754577)&(batch4.mouse != 754575)]

# Import data from batch3
batch3 = pd.read_csv(os.path.join(data_path,  'batch_3.csv'))
batch3 = batch3.loc[(batch3.mouse != 715866)&(batch3.mouse != 713578)]

# Merge both datasets
df = pd.concat([batch3, batch4], ignore_index=True)

df= df.loc[~df.patch_label.isin(['patch_delayed', 'patch_no_reward', 'patch_single', 'delayed', 'single', 'no_reward', 'PatchZB'])]

# df['patch_label'] = df['patch_label'].replace({'Alpha pinene': '60','Alpha-pinene': '60', 'Methyl Butyrate': '90', 'Ethyl Butyrate': '90', 'Amyl Acetate': '0', 
#                                                '2,3-Butanedione': 'slow', '2-Heptanone': 'slow',  'Methyl Acetate':'fast', 'Fenchone':'0'})

max_prob = df.groupby(['session', 'odor_label'])['reward_probability'].transform('max')
df['patch_label'] = max_prob.apply(lambda x: '90' if x > 0.75 else ('60' if x > 0.3 else '0'))

df['experiment'] = df['experiment'].replace({'base': 'control'})
df = df.loc[(df.experiment =='control')|(df.experiment =='data_collection')]
df['experiment'] = df['experiment'].replace({'data_collection': 'control'})

In [ ]:
# listA = df.session.unique()
# pd.DataFrame(listA, columns=['session']).to_csv(os.path.join(data_path, f'sessions_data_collection_3_4_list.csv'), index=False)

In [ ]:
## Subsampling of sessions for DDM
filt_sessions  = pd.read_csv(os.path.join(data_path, 'filtered_sessions.csv'), header = None)
lookup = pd.read_csv(os.path.join(data_path, 'session_lookup_return.csv'))

# map bad names to good names in df1
lookup = lookup.drop_duplicates(subset='aws_name')
filt_sessions['good_name'] = filt_sessions[0].map(lookup.set_index('aws_name')['input_session'])

# filter df3 where session is in the good names
df.session = df.session.str.split('_').str[-1]
df['session'] = df['mouse'].astype(str) + '_' + df['session']
df = df[df['session'].isin(filt_sessions['good_name'])]
print(df.session.nunique())

In [ ]:
groups = ['session','mouse','patch_number','patch_label','experiment']

pre_df = df.loc[(df['patch_number'] <50)&(df['site_number'] > 0)&(df['last_site'] == 1)].copy()

# These df summarizes each patch for each session for each mouse
mouse_df = (
    pre_df
    .groupby(['session','mouse','patch_number','patch_label','experiment'])
    .agg(
        site_number=('site_number', 'max'),
        reward_probability=('reward_probability', 'min'),
        stops=('site_number', 'max'),
        total_rewards=('cumulative_rewards', 'max'),
        consecutive_rewards = ('consecutive_rewards', 'max'),
        total_failures=('cumulative_failures', 'max'),
        consecutive_failures = ('consecutive_failures', 'max'), 
    )
    .reset_index()
)
mouse_df['total_water'] = mouse_df['total_rewards']*5
groups.pop(groups.index('patch_number'))

# These df summarizes each session for each mouse (averages patches within session)
session_df = ( 
        mouse_df
        .groupby(groups)
        .agg(site_number = ('site_number','sum'), 
              reward_probability = ('reward_probability','mean'), 
              stops = ('stops','mean'),
              total_stops = ('stops','sum'), 
              total_rewards = ('total_rewards','mean'),
              consecutive_rewards = ('consecutive_rewards','mean'),
              total_failures = ('total_failures','mean'),
              consecutive_failures = ('consecutive_failures','mean'), 
              patch_number = ('patch_number','nunique'), 
              total_water = ('total_water','sum'))
        .reset_index()
)

groups.pop(groups.index('session'))

# These df summarizes metrics for each mouse (averages all sessions and all patches withing that session)
general_df = ( 
        session_df
        .groupby(['mouse','patch_label', 'experiment'])
        .agg({'site_number':'mean', 
              'reward_probability':'mean', 
              'stops':'mean', 
              'total_rewards':'mean',
              'consecutive_rewards':'mean',
              'total_failures':'mean',
              'consecutive_failures':'mean', 
              'patch_number':'mean'
              })
        .reset_index()
)

In [ ]:
groups = ['mouse','patch_label','experiment']

mouse_df['total_water'] = mouse_df['total_rewards']*5

# These df summarizes each session for each mouse (averages patches within session)
mega_df = ( 
        mouse_df
        .groupby(groups)
        .agg(site_number = ('site_number','sum'), 
              reward_probability = ('reward_probability','mean'), 
              stops = ('stops','mean'),
              total_stops = ('stops','sum'), 
              total_rewards = ('total_rewards','mean'),
              consecutive_rewards = ('consecutive_rewards','mean'),
              total_failures = ('total_failures','mean'),
              consecutive_failures = ('consecutive_failures','mean'), 
              patch_number = ('patch_number','nunique'), 
              total_water = ('total_water','sum'))
        .reset_index()
)

In [ ]:
plot_reward_probability(general_df, experiment="control", 
                        condition='mouse', odor_labels=['90','60'], save=os.path.join(results_path, f'reward_probability_by_patch_.pdf'))

In [ ]:
plot_reward_probability(mega_df, experiment='control', 
                        condition='mouse', odor_labels=['90','60'], save=os.path.join(results_path, f'reward_probability_by_patch_.pdf'))

In [ ]:
# for experiment in general_df.experiment.unique():
with PdfPages(results_path+f'/summary_general_results_control_all.pdf') as pdf:
    f.summary_main_variables(mega_df, condition='mouse', experiment='control',save=pdf, odor_labels=['90','60'])

In [ ]:
mouse = 754559
plot_reward_probability(session_df.loc[session_df.mouse == mouse], experiment='control', 
                        condition='session', odor_labels=['90','60'], save=os.path.join(results_path, f'reward_probability_by_patch_{mouse}.pdf'))